# 6.3 Scikit-Learn to ONNX — Apply Notebook

## Objective

Convert scikit-learn estimators and pipelines to ONNX using **skl2onnx**,
verify prediction parity, and compare performance.

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Convert LogisticRegression | `convert_sklearn`, `FloatTensorType` |
| 2 | Convert Pipeline (StandardScaler + RandomForest) | Composite estimator conversion |
| 3 | Probability prediction parity | `predict_proba` vs ORT output |
| 4 | Convert GradientBoostingClassifier | Tree-ensemble conversion |
| 5 | Inspect converted model structure | Nodes, ops, initializers |
| 6 | Compare model sizes (pickle vs ONNX) | Serialization efficiency |
| 7 | Benchmark: sklearn predict vs ORT | Latency comparison |
| 8 | **Challenge:** convert a regression pipeline | `SVR` + pipeline |

```
pip install scikit-learn skl2onnx onnx onnxruntime numpy
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, tempfile, warnings, pickle
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import accuracy_score

import onnx
from onnx import checker, TensorProto
import onnxruntime as ort

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    HAS_SKL2ONNX = True
except ImportError:
    HAS_SKL2ONNX = False
    print("skl2onnx not installed.  Install with:  pip install skl2onnx")

import sklearn
print(f"scikit-learn : {sklearn.__version__}")
print(f"ONNX         : {onnx.__version__}")
print(f"ORT          : {ort.__version__}")
if HAS_SKL2ONNX:
    import skl2onnx
    print(f"skl2onnx     : {skl2onnx.__version__}")
print(f"NumPy        : {np.__version__}")

_SKIP_MSG = "⚠ skl2onnx not available — skipping. Install with: pip install skl2onnx"

## Exercise 1 — Convert LogisticRegression to ONNX

Logistic Regression models the probability of class $k$ as:

$$P(y{=}k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k^\top \mathbf{x} + b_k}}{\sum_{j} e^{\mathbf{w}_j^\top \mathbf{x} + b_j}}$$

`skl2onnx` needs an **initial type** declaration that specifies the input
tensor's name, element type, and shape.  `FloatTensorType([None, n_features])`
creates a float32 input with dynamic batch.

In [ ]:
if HAS_SKL2ONNX:
    np.random.seed(0)
    X_cls, y_cls = make_classification(
        n_samples=500, n_features=8, n_informative=6,
        n_redundant=1, n_classes=3, random_state=0,
    )
    X_cls = X_cls.astype(np.float32)

    lr = LogisticRegression(max_iter=1000, random_state=0)
    lr.fit(X_cls, y_cls)
    sk_acc = accuracy_score(y_cls, lr.predict(X_cls))
    print(f"LogisticRegression train accuracy: {sk_acc:.4f}")

    initial_type = [("X", FloatTensorType([None, X_cls.shape[1]]))]
    onnx_lr = convert_sklearn(lr, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_lr)

    print(f"ONNX nodes : {len(onnx_lr.graph.node)}")
    print(f"Ops        : {sorted(set(n.op_type for n in onnx_lr.graph.node))}")

    # Run via ORT
    sess = ort.InferenceSession(onnx_lr.SerializeToString(), providers=["CPUExecutionProvider"])
    ort_pred = sess.run(None, {"X": X_cls[:20]})

    ort_labels = ort_pred[0]
    sk_labels  = lr.predict(X_cls[:20])
    assert np.array_equal(ort_labels, sk_labels), "Label mismatch!"
    print(f"Label parity (first 20): ✓  — all {len(sk_labels)} match")
else:
    print(_SKIP_MSG)

## Exercise 2 — Convert a Pipeline (StandardScaler + RandomForest)

sklearn `Pipeline` chains a **preprocessor** and a **classifier**.  `skl2onnx`
converts the entire pipeline in one call — the scaler becomes ONNX arithmetic
ops (subtract mean, divide by std), and the forest becomes a `TreeEnsembleClassifier`.

$$\hat{\mathbf{x}} = \frac{\mathbf{x} - \boldsymbol{\mu}}{\boldsymbol{\sigma}}$$

In [ ]:
if HAS_SKL2ONNX:
    X_pipe, y_pipe = make_classification(
        n_samples=800, n_features=12, n_informative=10,
        n_classes=2, random_state=1,
    )
    X_pipe = X_pipe.astype(np.float32)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=50, max_depth=8, random_state=0)),
    ])
    pipe.fit(X_pipe, y_pipe)

    initial_type = [("X", FloatTensorType([None, X_pipe.shape[1]]))]
    onnx_pipe = convert_sklearn(pipe, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_pipe)

    print(f"Pipeline ONNX nodes: {len(onnx_pipe.graph.node)}")
    print(f"Ops: {sorted(set(n.op_type for n in onnx_pipe.graph.node))}")

    sess = ort.InferenceSession(onnx_pipe.SerializeToString(), providers=["CPUExecutionProvider"])

    X_test = X_pipe[:100]
    sk_pred = pipe.predict(X_test)
    ort_out = sess.run(None, {"X": X_test})
    ort_pred = ort_out[0]

    match_rate = np.mean(sk_pred == ort_pred)
    print(f"\nLabel match rate: {match_rate:.4f}")
    assert match_rate > 0.99, f"Match rate too low: {match_rate}"
    print(f"sklearn accuracy: {accuracy_score(y_pipe[:100], sk_pred):.4f}")
    print("Pipeline conversion verified ✓")
else:
    print(_SKIP_MSG)

## Exercise 3 — Probability Prediction Parity

For classifiers, `skl2onnx` emits **two** outputs:
1. Predicted label (int64)
2. Probability map (a list of dicts, or a 2-D array depending on the converter)

We extract the probability matrix and compare against `predict_proba`.

$$\text{max}_{i} \left| P_{\text{sklearn}}(y{=}c_i \mid \mathbf{x}) - P_{\text{ORT}}(y{=}c_i \mid \mathbf{x}) \right|$$

In [ ]:
if HAS_SKL2ONNX:
    def extract_proba(ort_outputs, n_samples):
        """Extract probability matrix from skl2onnx ORT outputs."""
        for arr in ort_outputs:
            if isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[0] == n_samples:
                return arr
            if isinstance(arr, list) and len(arr) == n_samples and isinstance(arr[0], dict):
                classes = sorted(arr[0].keys())
                return np.array([[d[c] for c in classes] for d in arr], dtype=np.float32)
        raise RuntimeError("Could not extract probability matrix")

    X_test = X_cls[:50]

    # LogisticRegression probabilities
    sk_proba = lr.predict_proba(X_test)
    sess_lr = ort.InferenceSession(onnx_lr.SerializeToString(), providers=["CPUExecutionProvider"])
    ort_out = sess_lr.run(None, {"X": X_test})
    ort_proba = extract_proba(ort_out, len(X_test))

    diff_lr = np.abs(sk_proba - ort_proba).max()
    print(f"LogisticRegression  predict_proba max|Δ|: {diff_lr:.2e}")
    assert diff_lr < 1e-4, f"LR proba diff too large: {diff_lr}"

    # Pipeline (RandomForest) probabilities
    sk_proba_pipe = pipe.predict_proba(X_pipe[:50])
    sess_pipe = ort.InferenceSession(onnx_pipe.SerializeToString(), providers=["CPUExecutionProvider"])
    ort_out_pipe = sess_pipe.run(None, {"X": X_pipe[:50]})
    ort_proba_pipe = extract_proba(ort_out_pipe, 50)

    diff_pipe = np.abs(sk_proba_pipe - ort_proba_pipe).max()
    print(f"RF Pipeline         predict_proba max|Δ|: {diff_pipe:.2e}")
    assert diff_pipe < 1e-4, f"Pipeline proba diff too large: {diff_pipe}"

    # Verify argmax matches predict
    ort_argmax = ort_proba_pipe.argmax(axis=1)
    sk_labels_pipe = pipe.predict(X_pipe[:50])
    match = np.mean(ort_argmax == sk_labels_pipe)
    print(f"argmax(proba) vs predict match: {match:.4f}")
    print("Probability parity verified ✓")
else:
    print(_SKIP_MSG)

## Exercise 4 — Convert GradientBoostingClassifier

Gradient Boosting builds an additive ensemble of shallow trees:

$$F_M(\mathbf{x}) = F_0 + \sum_{m=1}^{M} \eta \cdot h_m(\mathbf{x})$$

where $\eta$ is the learning rate and $h_m$ are regression trees fitted to
the negative gradient of the loss.  `skl2onnx` maps this to the
`TreeEnsembleClassifier` ONNX ML operator.

In [ ]:
if HAS_SKL2ONNX:
    X_gb, y_gb = make_classification(
        n_samples=600, n_features=10, n_informative=8,
        n_classes=3, random_state=42,
    )
    X_gb = X_gb.astype(np.float32)

    gbc = GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=0,
    )
    gbc.fit(X_gb, y_gb)
    sk_acc = accuracy_score(y_gb, gbc.predict(X_gb))
    print(f"GBC train accuracy: {sk_acc:.4f}")

    initial_type = [("X", FloatTensorType([None, X_gb.shape[1]]))]
    onnx_gbc = convert_sklearn(gbc, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_gbc)

    sess = ort.InferenceSession(onnx_gbc.SerializeToString(), providers=["CPUExecutionProvider"])

    X_test = X_gb[:100]
    sk_pred = gbc.predict(X_test)
    ort_pred = sess.run(None, {"X": X_test})[0]
    match = np.mean(sk_pred == ort_pred)

    sk_proba = gbc.predict_proba(X_test)
    ort_proba = extract_proba(sess.run(None, {"X": X_test}), len(X_test))
    proba_diff = np.abs(sk_proba - ort_proba).max()

    print(f"Label match rate : {match:.4f}")
    print(f"Proba max|Δ|     : {proba_diff:.2e}")
    print(f"ONNX nodes       : {len(onnx_gbc.graph.node)}")
    print(f"Ops              : {sorted(set(n.op_type for n in onnx_gbc.graph.node))}")
    assert match > 0.99
    print("GBC conversion verified ✓")
else:
    print(_SKIP_MSG)

## Exercise 5 — Inspect Converted Model Structure

sklearn converters produce domain-specific ONNX ML operators like
`TreeEnsembleClassifier` and `LinearClassifier` rather than the standard
math ops you see in neural-network exports.  Let's look inside.

In [ ]:
if HAS_SKL2ONNX:
    from collections import Counter

    models_to_inspect = [
        ("LogisticRegression", onnx_lr),
        ("RF Pipeline", onnx_pipe),
        ("GradientBoosting", onnx_gbc),
    ]

    for name, proto in models_to_inspect:
        g = proto.graph
        init_names = {i.name for i in g.initializer}

        print(f"\n{'═'*50}")
        print(f"  {name}")
        print(f"{'═'*50}")

        print(f"  Inputs: ", end="")
        for inp in g.input:
            if inp.name not in init_names:
                dims = [d.dim_param or str(d.dim_value) for d in inp.type.tensor_type.shape.dim]
                print(f"{inp.name} [{', '.join(dims)}]")

        print(f"  Outputs:")
        for o in g.output:
            print(f"    {o.name}")

        counts = Counter(n.op_type for n in g.node)
        print(f"  Nodes: {len(g.node)}")
        for op, c in counts.most_common():
            print(f"    {op:<30} {c}")

        total_bytes = len(proto.SerializeToString())
        print(f"  Serialized size: {total_bytes/1024:.1f} KB")
else:
    print(_SKIP_MSG)

## Exercise 6 — Compare Model Sizes (Pickle vs ONNX)

sklearn's default serialization is `pickle` (or `joblib`).  ONNX uses
Protocol Buffers.  Let's compare file sizes.

$$\text{compression ratio} = \frac{\text{size}_{\text{pickle}}}{\text{size}_{\text{ONNX}}}$$

In [ ]:
if HAS_SKL2ONNX:
    models_for_size = [
        ("LogisticRegression", lr, onnx_lr),
        ("RF Pipeline", pipe, onnx_pipe),
        ("GradientBoosting", gbc, onnx_gbc),
    ]

    print(f"{'Model':<24} {'Pickle (KB)':>12} {'ONNX (KB)':>12} {'Ratio':>8}")
    print("─" * 60)

    for name, sk_model, onnx_model in models_for_size:
        pkl_bytes = len(pickle.dumps(sk_model))
        onnx_bytes = len(onnx_model.SerializeToString())

        ratio = pkl_bytes / onnx_bytes if onnx_bytes > 0 else float("inf")
        print(f"{name:<24} {pkl_bytes/1024:>10.1f} {onnx_bytes/1024:>10.1f} {ratio:>7.2f}x")

    print("\nRatio > 1 means pickle is larger than ONNX.")
else:
    print(_SKIP_MSG)

## Exercise 7 — Benchmark: sklearn `predict` vs ORT

We measure latency for prediction across different sample counts.

$$\text{speedup} = \frac{t_{\text{sklearn}}}{t_{\text{ORT}}}$$

ORT's tree-ensemble kernel is often significantly faster than sklearn's
Python-based implementation, especially for large forests.

In [ ]:
if HAS_SKL2ONNX:
    def bench(fn, warmup=20, iters=100):
        for _ in range(warmup):
            fn()
        times = []
        for _ in range(iters):
            t0 = time.perf_counter()
            fn()
            times.append(time.perf_counter() - t0)
        arr = np.array(times) * 1000
        return {"median": np.median(arr), "p95": np.percentile(arr, 95)}

    sess_bench = ort.InferenceSession(
        onnx_pipe.SerializeToString(), providers=["CPUExecutionProvider"]
    )

    print(f"Benchmark: RF Pipeline (50 trees, depth 8)")
    print(f"{'N samples':>10} {'sklearn (ms)':>14} {'ORT (ms)':>14} {'Speedup':>10}")
    print("─" * 52)

    for n in [10, 100, 500, 1000, 5000]:
        X_b = np.random.randn(n, 12).astype(np.float32)

        sk_stats  = bench(lambda: pipe.predict(X_b))
        ort_stats = bench(lambda: sess_bench.run(None, {"X": X_b}))
        speedup = sk_stats["median"] / ort_stats["median"]

        print(f"{n:>10} {sk_stats['median']:>11.3f} ms {ort_stats['median']:>11.3f} ms {speedup:>9.2f}x")

    print("\n(Speedup > 1 means ORT is faster)")
else:
    print(_SKIP_MSG)

## Exercise 8 — Challenge: Convert a Regression Pipeline

Convert a pipeline with `StandardScaler` + `SVR` (Support Vector Regressor)
for a regression task.  Verify that the **continuous predictions** match.

The SVR decision function is:

$$f(\mathbf{x}) = \sum_{i \in SV} \alpha_i \, K(\mathbf{x}_i, \mathbf{x}) + b$$

where $K$ is the kernel (RBF by default) and $SV$ is the set of support vectors.

In [ ]:
if HAS_SKL2ONNX:
    X_reg, y_reg = make_regression(
        n_samples=400, n_features=6, noise=0.5, random_state=7,
    )
    X_reg = X_reg.astype(np.float32)
    y_reg = y_reg.astype(np.float32)

    reg_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf", C=10.0)),
    ])
    reg_pipe.fit(X_reg, y_reg)

    initial_type = [("X", FloatTensorType([None, X_reg.shape[1]]))]
    onnx_reg = convert_sklearn(reg_pipe, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_reg)

    print(f"Regression pipeline converted — {len(onnx_reg.graph.node)} nodes")
    print(f"Ops: {sorted(set(n.op_type for n in onnx_reg.graph.node))}")

    # Parity on continuous predictions
    sess = ort.InferenceSession(onnx_reg.SerializeToString(), providers=["CPUExecutionProvider"])
    X_test = X_reg[:50]
    sk_pred = reg_pipe.predict(X_test).astype(np.float32)
    ort_pred = sess.run(None, {"X": X_test})[0].flatten()

    max_diff = np.abs(sk_pred - ort_pred).max()
    mean_diff = np.abs(sk_pred - ort_pred).mean()
    r2 = 1 - np.sum((sk_pred - ort_pred)**2) / np.sum((sk_pred - sk_pred.mean())**2 + 1e-10)

    print(f"\nContinuous prediction parity:")
    print(f"  max|Δ|  : {max_diff:.2e}")
    print(f"  mean|Δ| : {mean_diff:.2e}")
    print(f"  R² (ORT vs sklearn): {r2:.6f}")
    assert max_diff < 1e-3, f"Regression parity failed: {max_diff}"
    print("Regression pipeline parity verified ✓")

    # Size comparison
    pkl_kb = len(pickle.dumps(reg_pipe)) / 1024
    onnx_kb = len(onnx_reg.SerializeToString()) / 1024
    print(f"\nSize — pickle: {pkl_kb:.1f} KB,  ONNX: {onnx_kb:.1f} KB")
else:
    print(_SKIP_MSG)

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| LogisticRegression | `convert_sklearn` + `FloatTensorType` for basic classifiers |
| Pipeline | Entire pipeline converts in one call; scaler → math ops, forest → `TreeEnsembleClassifier` |
| Probabilities | skl2onnx emits labels + probabilities; extract the 2-D array |
| GradientBoosting | 100+ tree boosting models convert cleanly to ONNX ML ops |
| Inspection | sklearn converters use domain-specific `ai.onnx.ml` operators |
| Size | ONNX can be larger or smaller than pickle depending on the model |
| Performance | ORT's C++ tree kernel often beats sklearn's Python predict |
| Regression | Continuous-output pipelines (SVR, LinearRegression, etc.) also convert |